# Baseline Features (Demographics + Healthcare Utilisation)

Builds the baseline predictor set -- age, and the five baseline healthcare-utilisation measures (number of diagnoses, procedures, claims, episodes, and length of baseline history) -- from the row-level claims data, restricted to the baseline window (on or before each patient's diabetes diagnosis date) per the Feature Engineering section of the Methodology.

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Load Data

In [2]:
DATA_DIR = './'  # raw cohort export CSVs (not included in this repo)

full = pd.read_csv(DATA_DIR + 't2d_cvd_cohort_full_v2.csv',
                    parse_dates=['diagnosed_time', 'index_date', 'cvd_date'])
summary = pd.read_csv(DATA_DIR + 't2d_cvd_cohort_v2.csv',
                       parse_dates=['index_date', 'cvd_date'])

print(f'Full row-level file: {len(full):,} rows, {full["person_id"].nunique():,} patients')
print(f'Summary file: {len(summary):,} patients')
print(f'Median follow-up: {summary["years_followup"].median():.2f} years')

Full row-level file: 108,860 rows, 3,050 patients
Summary file: 3,050 patients
Median follow-up: 2.89 years


## 2. Baseline Window

Keep only rows dated on or before each patient's `index_date` (same calendar day allowed, nothing after) -- unchanged from v1.

In [3]:
baseline = full[full['diagnosed_time'].dt.date <= full['index_date'].dt.date].copy()
print(f'Baseline rows: {len(baseline):,} ({100*len(baseline)/len(full):.1f}% of all rows)')
print(f'Patients with at least one baseline row: {baseline["person_id"].nunique():,} / {full["person_id"].nunique():,}')

Baseline rows: 34,156 (31.4% of all rows)
Patients with at least one baseline row: 3,050 / 3,050


## 3. Comorbidity Features

One flag per comorbidity category present in `chronic_condition` during the baseline window. `Diabetes` and `CVD` excluded (exposure and outcome, not predictors).

In [4]:
EXCLUDE_TAGS = {'', 'Diabetes', 'CVD'}

cc_baseline = baseline[['person_id', 'chronic_condition']].dropna(subset=['chronic_condition']).copy()
cc_baseline['chronic_condition'] = cc_baseline['chronic_condition'].fillna('')
cc_baseline = cc_baseline.assign(tag=cc_baseline['chronic_condition'].str.split('|')).explode('tag')
cc_baseline = cc_baseline[~cc_baseline['tag'].isin(EXCLUDE_TAGS)]

comorbidity_flags = (
    cc_baseline.assign(present=1)
    .pivot_table(index='person_id', columns='tag', values='present', aggfunc='max', fill_value=0)
)
comorbidity_flags.columns = [f'comorbid_{c}' for c in comorbidity_flags.columns]
print(f'Comorbidity columns: {list(comorbidity_flags.columns)}')
comorbidity_flags.head()

Comorbidity columns: ['comorbid_Alcohol', 'comorbid_Anemia', 'comorbid_Arthritis', 'comorbid_Asthma', 'comorbid_BackPain', 'comorbid_BloodLoss', 'comorbid_CKD', 'comorbid_COPD', 'comorbid_Cancer', 'comorbid_Coagulopathy', 'comorbid_Drugs', 'comorbid_FluidsLytes', 'comorbid_Gout', 'comorbid_Hypothyroid', 'comorbid_Liver', 'comorbid_MHC', 'comorbid_NeuroOther', 'comorbid_Obesity', 'comorbid_Osteo', 'comorbid_PUD', 'comorbid_Paralysis', 'comorbid_WeightLoss']


,comorbid_Alcohol,comorbid_Anemia,comorbid_Arthritis,comorbid_Asthma,comorbid_BackPain,comorbid_BloodLoss,comorbid_CKD,comorbid_COPD,comorbid_Cancer,comorbid_Coagulopathy,comorbid_Drugs,comorbid_FluidsLytes,comorbid_Gout,comorbid_Hypothyroid,comorbid_Liver,comorbid_MHC,comorbid_NeuroOther,comorbid_Obesity,comorbid_Osteo,comorbid_PUD,comorbid_Paralysis,comorbid_WeightLoss
person_id,,,,,,,,,,,,,,,,,,,,,,
686,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1035,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1219,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1247,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1277,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


## 4. Healthcare Utilisation Features

In [5]:
util = baseline.groupby('person_id').agg(
    baseline_n_diagnoses=('diagnosis_procedure_type', lambda s: (s == 'D').sum()),
    baseline_n_procedures=('diagnosis_procedure_type', lambda s: (s == 'P').sum()),
    baseline_n_claims=('claim_id', 'nunique'),
    baseline_n_episodes=('episode_id', 'nunique'),
    baseline_first_record=('diagnosed_time', 'min'),
)
util['baseline_history_days'] = (
    full.groupby('person_id')['index_date'].first() - util['baseline_first_record']
).dt.days
util = util.drop(columns=['baseline_first_record'])
util.head()

,baseline_n_diagnoses,baseline_n_procedures,baseline_n_claims,baseline_n_episodes,baseline_history_days
person_id,,,,,
438,18,0,2,2,4430
683,4,2,1,1,0
686,12,4,3,3,0
939,4,2,1,1,0
951,17,10,5,5,0


## 5. Assemble Baseline Feature Table

In [6]:
demo = summary.set_index('person_id')[['sex', 'age', 'postcode', 'incident_cvd', 'years_followup']]

baseline_features = demo.join(comorbidity_flags, how='left').join(util, how='left')

comorbid_cols = [c for c in baseline_features.columns if c.startswith('comorbid_')]
baseline_features[comorbid_cols] = baseline_features[comorbid_cols].fillna(0).astype(int)

util_cols = [c for c in util.columns]
baseline_features[util_cols] = baseline_features[util_cols].fillna(0)

baseline_features = baseline_features.reset_index()
print(f'Baseline feature table: {baseline_features.shape[0]:,} patients x {baseline_features.shape[1]} columns')
baseline_features.head()

Baseline feature table: 3,050 patients x 33 columns


,person_id,sex,age,postcode,incident_cvd,years_followup,comorbid_Alcohol,comorbid_Anemia,comorbid_Arthritis,comorbid_Asthma,comorbid_BackPain,comorbid_BloodLoss,comorbid_CKD,comorbid_COPD,comorbid_Cancer,comorbid_Coagulopathy,comorbid_Drugs,comorbid_FluidsLytes,comorbid_Gout,comorbid_Hypothyroid,comorbid_Liver,comorbid_MHC,comorbid_NeuroOther,comorbid_Obesity,comorbid_Osteo,comorbid_PUD,comorbid_Paralysis,comorbid_WeightLoss,baseline_n_diagnoses,baseline_n_procedures,baseline_n_claims,baseline_n_episodes,baseline_history_days
0,438,F,95,"5,044.00",True,0.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,0,2,2,4430
1,683,M,106,"2,001.00",False,16.15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,2,1,1,0
2,686,F,102,"2,073.00",False,18.54,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,12,4,3,3,0
3,939,F,102,"2,250.00",True,0.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,2,1,1,0
4,951,M,101,"4,165.00",False,18.54,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,17,10,5,5,0


In [7]:
OUT_FEATURES = 'baseline_features.csv'
baseline_features.to_csv(OUT_FEATURES, index=False)
print(f'Saved to: {OUT_FEATURES}')

Saved to: baseline_features.csv
